In [ ]:
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from collections import defaultdict
from modules.Plotting import MetricLog, plot_metrics
from modules.HandTesting import hand_test_repl

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer
from torch.optim import AdamW

import warnings
warnings.filterwarnings("ignore")

TOKEN_MODEL_NAME = 'token_model'
LINE_MODEL_NAME = "line_model_T5_small"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

ImportError: attempted relative import with no known parent package

[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Tokenizer

In [ ]:
# character-level BPE-lite

SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}

class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set(",;&|~^@#")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,_ \t\n":  #for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            for ch in indent:
                tokens.append(ch)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if ch == "\n" or ch.strip():
                        if buf:
                            tokens.append(buf)
                        tokens.append(ch)
                        continue
                    if buf:
                        buf += ch
                        tokens.append(buf)
                        buf = ""
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)

## Datasets

In [ ]:

def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts



class TokenDataset(Dataset):
    """
    Sliding-window dataset for next-token prediction.
    Target at each position is the next token id.
    """
    def __init__(self, ids: List[int], ctx: int = 128):
        self.ctx = ctx
        self.data = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return max(0, len(self.data) - self.ctx - 1)

    def __getitem__(self, i):
        x = self.data[i: i + self.ctx]
        y = self.data[i + 1: i + self.ctx + 1]
        return x, y

## Models

In [ ]:
@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class TokenModel(nn.Module):
    """
    Decoder-only Transformer for causal next-token prediction.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.emb   = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.pos   = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        layer      = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,
            batch_first=True, norm_first=True
        )
        self.enc   = nn.TransformerEncoder(layer, cfg.n_layers)
        self.head  = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.emb.weight = self.head.weight  # weight tying

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.pos(self.emb(x))
        h = self.enc(h, mask=mask, is_causal=True)
        return self.head(h)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int,
                 temperature: float = 0.8, top_k: int = 50,
                 stop_at_word_end: bool = True,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev   = next(self.parameters()).device
        ids   = list(prefix_ids)
        generated = []
        PUNCT_CHARS = set("()[]{}.,;:=+-*/\\%<>!&|~^@# \t\n\"'`")
        for _ in range(max_new):
            x = torch.tensor([ids[-self.cfg.max_len:]], dtype=torch.long, device=dev)
            logits = self(x)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            ids.append(nxt)
            generated.append(nxt)
            if stop_at_word_end and tokenizer:
                tok = tokenizer.id2token.get(nxt, "")
                if any(c in PUNCT_CHARS for c in tok):
                    break
        return generated

## Training loop

In [ ]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(
    model:      TokenModel,
    train_dl:   DataLoader,
    val_dl:     DataLoader,
    epochs:     int,
    lr:         float,
    device:     torch.device,
    saver:      BestModelSaver,
    log:        MetricLog,
    plot_dir:   str,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for x, y in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                 leave=False, unit="batch"):
            # if index % one_part == 0:
                # print(f"{index // one_part}% at {datetime.now().time()}")
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds = logits.argmax(-1)
            mask = (y != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == y[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for x, y in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                 leave=False, unit="batch"):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        print(f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
              f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"./{plot_dir}/{TOKEN_MODEL_NAME}_ep{ep:02d}.png")

    plot_metrics(log, f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Final", f"./{plot_dir}/{TOKEN_MODEL_NAME}_final.png")

## Hand Testing

In [ ]:
def hand_test_repl(token_model: TokenModel, line_model: T5ForConditionalGeneration,
                   tokenizer: CodeTokenizer, hf_tok: AutoTokenizer,
                   device: torch.device):
    print("  Python Autocomplete — Interactive Test")
    print("  Commands: :token <prefix>  |  :line <prefix>")
    print("            :temp <float>   |  :k <int>  |  :quit")

    temperature = 0.3    # lower default — better for code
    top_k       = 10

    while True:
        try:
            raw = input(">> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye!")
            break

        if not raw:
            continue

        if raw.startswith(":quit"):
            break
        elif raw.startswith(":temp"):
            try:   temperature = float(raw.split()[1])
            except: print("Usage: :temp 0.7")
            print(f"temperature = {temperature}")
            continue
        elif raw.startswith(":k"):
            try:   top_k = int(raw.split()[1])
            except: print("Usage: :k 40")
            print(f"top_k = {top_k}")
            continue
        elif raw.startswith(":token"):
            prefix = raw[6:]
            ids = tokenizer.encode(prefix)[:-1]
            with torch.no_grad():
                new_ids = token_model.generate(
                    ids, max_new=20, temperature=temperature,
                    top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")
        else:
            prefix = raw[]
            ids = tokenizer.encode(prefix)[:-1]
            with torch.no_grad():
                new_ids = token_model.generate(
                    ids, max_new=20, temperature=temperature,
                    top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")

## Main

In [ ]:
class Arguments():
    def __init__(self, data_dir: str = "./Clean_Dataset", ckpt_dir: str = "./checkpoints",
                    plot_dir: str = "plots", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    args = Arguments()
    # args = Arguments(max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    torch.serialization.add_safe_globals([ModelCfg])

    HF_MODEL = "Salesforce/codet5-small"
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        tm = TokenModel(cfg).to(device)
        tok_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / TOKEN_MODEL_NAME + "_*.pt")))
        if tok_paths:
            ck = torch.load(tok_paths[0], map_location=device, weights_only=False)
            tm.load_state_dict(ck["model_state"])
            print(f"[Loaded] token model from {tok_paths[0]}")
    
        lm = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME + "_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(tm, lm, tokenizer, hf_tok, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── TOKEN MODEL ──────────────────────────────────────────
        print("  Prepairing TOKEN model")

        # flatten all train text → single id stream
        all_ids_tr = []
        for t in tr_txt:
            all_ids_tr.extend(tokenizer.encode(t))
        all_ids_va = []
        for t in va_txt:
            all_ids_va.extend(tokenizer.encode(t))

        tr_ds = TokenDataset(all_ids_tr, args.ctx)
        va_ds = TokenDataset(all_ids_va, args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,  num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False, num_workers=0, pin_memory=True)

        tok_model = TokenModel(cfg).to(device)
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.2f}M parameters")

        tok_saver = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        tok_log   = MetricLog()
        print("  Training TOKEN model")
        train_token_model(tok_model, tr_dl, va_dl, args.epochs, args.lr,
                          device, tok_saver, tok_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(tok_model, tokenizer, hf_tok, device)


main()

[Tokenizer] building from data …
[Data] loaded 0 files from Clean_Dataset
[Tokenizer] vocab_size=4


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[Loading] Started loading
[Data] loaded 0 files from Clean_Dataset
[ERROR] no data files found. Please put .py files in --data_dir


C:\Programing\code_autocomplete\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
